In [1]:
from src.classes.crossword_puzzle import CrosswordPuzzle
from src.classes.guesses import Guess
from src.constants import CLUE_ID, PUZ_FILE_DIR
from src.prompts.get_guesses_for_clue_using_llm import get_guesses_for_clue_using_llm
from src.prompts.reorder_clues_using_llm import reorder_clues_using_llm

In [2]:
crossword_puzzle = CrosswordPuzzle(f"{PUZ_FILE_DIR}/nytm_2025_01_01.puz")
clues = reorder_clues_using_llm(crossword_puzzle.get_clues())
guesses: dict[CLUE_ID, list[Guess]] = {}

In [3]:
clues

[Clue(text='Stop', length=5, number=8, direction='across', row=3, col=0),
 Clue(text='Love so much', length=5, number=6, direction='across', row=1, col=0),
 Clue(text='Soccer contest', length=5, number=1, direction='down', row=0, col=0),
 Clue(text='Searches (for)', length=5, number=9, direction='across', row=4, col=0),
 Clue(text="Feature of a cockatoo's head", length=5, number=4, direction='down', row=0, col=3),
 Clue(text='Like Janus, the god of beginnings', length=5, number=3, direction='down', row=0, col=2),
 Clue(text="Francophile's farewell", length=5, number=2, direction='down', row=0, col=1),
 Clue(text='Maker of Ironman Triathlon watches', length=5, number=7, direction='across', row=2, col=0),
 Clue(text='Not-so-nice magic spells', length=5, number=5, direction='down', row=0, col=4),
 Clue(text='Month that was the first of the new year in early 3-Down calendars', length=5, number=1, direction='across', row=0, col=0)]

In [4]:
clue_index = 0

while not crossword_puzzle.is_solved:
    clue = clues[clue_index]
    print(f"Attempting to solve clue {clue.number} {clue.direction} - {clue.text}")

    if clue.id not in guesses:
        pattern = crossword_puzzle.get_pattern(clue)
        print(f"Generating guesses for clue: {clue.number} {clue.direction} - {clue.text} with pattern '{pattern}'")
        clue_guesses = get_guesses_for_clue_using_llm(clue ,pattern)
        guesses[clue.id] = clue_guesses

    clue_guesses = guesses[clue.id]

    if len(clue_guesses) == 0:
        print(f"No guesses left for clue: {clue.number} {clue.direction} - {clue.text}, backtracking...")
        guesses.pop(clue.id)

        if clue_index != 0:
            clue_index -= 1
            crossword_puzzle.remove_answer(clues[clue_index])

        continue

    print(f"Guesses for clue {clue.number} {clue.direction}:")
    for guess in clue_guesses:
        print(f" - {guess.answer} (confidence: {guess.confidence_score}): {guess.explanation}")

    best_guess = max(clue_guesses, key=lambda g: g.confidence_score)

    try:
        crossword_puzzle.set_answer(clue, best_guess.answer)
        clue_guesses.remove(best_guess)

        print(f"Set answer for clue {clue.number} {clue.direction} to '{best_guess.answer}'")
        clue_index += 1
    except Exception as e:

        clue_guesses.remove(best_guess)
        print(f"Error setting answer for clue {clue.number} {clue.direction}: {e}")
    finally:
        crossword_puzzle.print_grid()

Attempting to solve clue 8 across - Stop
Generating guesses for clue: 8 across - Stop with pattern '['_', '_', '_', '_', '_']'
Guesses for clue 8 across:
 - CEASE (confidence: 85): Another synonym for 'stop' is CEASE.
 - PAUSE (confidence: 80): To stop or halt movement is to PAUSE.
 - CHECK (confidence: 75): A synonym for 'stop' or 'check' is CHECK.
Set answer for clue 8 across to 'CEASE'
_ _ _ _ _
_ _ _ _ _
_ _ _ _ _
C E A S E
_ _ _ _ _
Attempting to solve clue 6 across - Love so much
Generating guesses for clue: 6 across - Love so much with pattern '['_', '_', '_', '_', '_']'


KeyboardInterrupt: 